# Faruq-v3 DC2 global–local MSFA screen

Frozen-detector validation screen for a YOLO26 adaptation of DC2 multi-stream feature aggregation. P3/P4/P5 predicted-box feature patches are globally pooled, projected to the MobileNetV3 local descriptor, and added residually. The comparison is against an optimization-matched local-only control. This is not a literal stage-paired reproduction and not yet end-to-end DC2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/dc2-msfa-global-local-screening'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)
print('BRANCH:', BRANCH)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-dc2-predicted-raw-crop-screening-v1/dc2_predicted_raw_crop_screening.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DETECTOR = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt')
DC2B_SUMMARY = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-dc2-predicted-raw-crop-screening-v1/dc2_predicted_raw_crop_screening.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-dc2-msfa-global-local-screening-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Locked holdout tidak boleh tersedia.'
print('GPU      :', torch.cuda.get_device_name(0))
print('PROJECT  :', PROJECT_ROOT)
print('DATA     :', DATA_ROOT)
print('DETECTOR :', DETECTOR)
print('DC2b     :', DC2B_SUMMARY)
print('OUTPUT   :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_dc2_crop.py',
    'tests/test_dc2_predicted_crop.py',
    'tests/test_dc2_msfa.py',
]
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('PASS: zero-init preservation, MSFA activation, cache signature, and frozen decision gate verified.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_dc2_msfa_screening',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--detector-checkpoint', str(DETECTOR),
    '--dc2b-summary', str(DC2B_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--epochs', '10', '--batch-size', '64', '--workers', '2',
    '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'DC2 MSFA screen gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'dc2_msfa_global_local_screening.json'
assert SUMMARY.is_file(), f'DC2 MSFA screen belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val_detector_matched_targets'
assert result['test_images_accessed'] is False
rows = []
for arm_name in ('LOCAL_FT', 'MSFA'):
    metrics = result['results'][arm_name]['metrics']
    rows.append({'arm': arm_name, **{k: metrics[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}})
frame = pd.DataFrame(rows)
display(frame.style.format({'accuracy': '{:.2%}', 'macro_f1': '{:.2%}', 'bottom3_f1': '{:.2%}', 'worst_f1': '{:.2%}'}))
print('EQ TRANSFER :', result['paper_equation_transferred'])
print('BOUNDARY    :', result['adaptation_boundary'])
print('STATIC      :', result['static_gates'])
print('DELTA       :', result['deltas_msfa_vs_local_ft'])
print('CRITERIA    :', result['criteria'])
print('DECISION    :', result['decision'])
print('NEXT        :', result['next_action'])
print('SUMMARY     :', SUMMARY)